In [1]:
from import_all import *
from utilities import *
import pyvisa

In [3]:
#----Connect to instrument----#
#load the station from yaml configuration file
station = Station(config_file="electrochemistry.station.yaml")
keithley1 = station.load_keithley1()
keithley2 = station.load_keithley2()
cha = keithley1.smua
chb = keithley1.smub
chc = keithley2.smua
chd = keithley2.smub

rm = pyvisa.ResourceManager()
intfc = rm.open_resource('GPIB0::INTFC')


Connected to: Keithley Instruments Inc. 2636B (serial:4629010, firmware:3.4.0) in 2.27s
Connected to: Keithley Instruments Inc. 2614B (serial:4070377, firmware:3.0.4) in 1.42s


In [4]:
channels = [cha,chb,chc,chd]
keithleys = [keithley1, keithley2]
# for ch in channels:
#     ch.write(f"{ch.channel}.reset()")


In [5]:
cha.nplc(25)
chb.nplc(25)
chc.nplc(25)
chd.nplc(25)

In [6]:
def source_trig_params(chan):
    #tie source to bus trigger
    chan.write(f"{chan.channel}.trigger.source.stimulus = trigger.EVENT_ID")

    #end of sweep phase action
    chan.write(f"{chan.channel}.trigger.endsweep.action = smua.SOURCE_HOLD")

    #enable
    chan.write(f"{chan.channel}.trigger.source.action = {chan.channel}.ENABLE")

    

In [7]:
def meas_trig_params(chan):
    #setup buffer
    chan.write(f"{chan.channel}.trigger.measure.i({chan.channel}.nvbuffer1)")
    chan.write(f"{chan.channel}.nvbuffer1.appendmode = 1")

    #clear any residual values
    chan.write(f"{chan.channel}.nvbuffer1.clear()")
    chan.write(f"{chan.channel}.nvbuffer1.collectsourcevalues = 1")

    #set measure trig to automatic (after source)
    chan.write(f"{chan.channel}.measure.count = 1")
    chan.write(f"{chan.channel}.trigger.measure.stimulus = 0")

    #enable
    chan.write(f"{chan.channel}.trigger.measure.action = {chan.channel}.ENABLE")


In [8]:
def trigger(keithleys, channels):
    for ch in channels:
        ch.write(f"{ch.channel}.nvbuffer1.clear()")
        ch.write(f"{ch.channel}.trigger.initiate()")

    for k in keithleys:
        k.write("*TRG")



In [9]:
def recall_buffer(ch):
    j = ch.ask(f"{ch.channel}.nvbuffer1.readings[1]") 
    v = ch.ask(f"{ch.channel}.nvbuffer1.sourcevalues[1]")
    return v, j


In [106]:
cha.write("smua.source.levelv = 0")
cha.write(f"smua.abort()")

chb.write("smub.source.levelv = 0")
chb.write(f"smub.abort()")

chc.write("smua.source.levelv = 0")
chc.write(f"smua.abort()")

chd.write("smub.source.levelv = 0")
chd.write(f"smub.abort()")

In [10]:
for ch in channels:
    source_trig_params(ch)
    meas_trig_params(ch)

In [11]:
v = [0.1,0.2]
js = []
vs = []
for volt in v:
    volt = str(volt)
    for ch in channels:
        ch.write(f"{ch.channel}.trigger.source.linearv({volt}, {volt}, 1)")

    trigger(keithleys, channels)
    for channel in channels:
        v, j = recall_buffer(channel)
        vs.append(v)
        js.append(j)

In [13]:
vs

['1.00000e-01',
 '1.00000e-01',
 '1.00000e-01',
 '1.00000e-01',
 '2.00000e-01',
 '2.00000e-01',
 '2.00000e-01',
 '2.00000e-01']

In [44]:
volt = "0.1"
for ch in channels:
    ch.write(f"{ch.channel}.trigger.source.linearv({volt}, {volt}, 1)")


In [45]:
for ch in channels:
    ch.write(f"{ch.channel}.trigger.initiate()")
    # intfc.assert_trigger()
    

In [46]:
keithley1.write("*TRG")
keithley2.write("*TRG")